# 04 - γ' Phase Candidate RankingRank L1₂ candidates for γ' phase potential using a weighted score.

In [ ]:
import syssys.path.insert(0, '..')from llm.sql_generator import pipelinefrom safety.sql_guard import execute_sqlimport pandas as pdimport numpy as np

In [ ]:
# Get all L1₂ data with propertiesquery = "L1₂型金属間化合物の安定性と弾性特性を表示して"result = pipeline(query)print(result["sql"])

In [ ]:
exec_result = execute_sql(result["sql"])if exec_result["success"]:    df = pd.DataFrame(exec_result["rows"], columns=exec_result["columns"])    display(df.head(10))

In [ ]:
# Composite scoring# score = w1*stability + w2*lattice_match + w3*bulk_modulus + w4*known_elementNI3AL_LATTICE = 3.572w1, w2, w3, w4 = 0.3, 0.3, 0.2, 0.2def compute_gamma_prime_score(row):    """Compute a composite gamma-prime candidate score."""    stability = max(0, 1.0 - row.get("energy_above_hull", 0.05) / 0.05)    lattice_a = row.get("lattice_a", NI3AL_LATTICE)    lattice_match = max(0, 1.0 - abs(lattice_a - NI3AL_LATTICE) / 0.5)    bm = row.get("bulk_modulus", 150) / 250.0    known_elems = {"Ni", "Co", "Al", "Ti"}    formula = str(row.get("formula", ""))    import re    elems = set(re.findall(r"[A-Z][a-z]?", formula))    known_score = len(elems & known_elems) / max(len(elems), 1)    return w1*stability + w2*lattice_match + w3*bm + w4*known_scoreprint("Scoring function defined")

In [ ]:
if exec_result["success"] and len(df) > 0:    df["gamma_prime_score"] = df.apply(compute_gamma_prime_score, axis=1)    df_ranked = df.sort_values("gamma_prime_score", ascending=False)    print("Top 20 gamma-prime candidates:")    display(df_ranked.head(20))    df_ranked.to_csv("../gamma_prime_candidate_ranking.csv", index=False)    print("Saved gamma_prime_candidate_ranking.csv")

## Materials Design Hypotheses

In [ ]:
hypotheses = """# L1₂ Materials Design Hypotheses## Key Findings1. **Ni-based L1₂ compounds** show the strongest stability (lowest energy above hull)   among A₃B-type intermetallics, consistent with the well-known γ' phase in Ni-base superalloys.2. **Co-based L1₂ compounds** (Co₃Al, Co₃Ti, Co₃Ta, Co₃W) represent promising   next-generation γ' phase candidates with competitive lattice match to Ni₃Al.3. **Lattice constant clustering** around 3.55-3.72 Å for stable L1₂ compounds   suggests geometric packing constraints that favor specific A-B element combinations.4. **Formation energy trends** indicate that early transition metals (Ti, Sc, Hf)   at the B-site generally produce more negative formation energies.5. **Bulk modulus** correlates with the atomic number of the A-site element:   5d elements (Ir, Pt) yield the highest elastic stiffness.## Design Guidelines- For γ' phase strengthening: prioritize Ni₃Al-like lattice constants (3.55-3.60 Å)- For high-temperature stability: target formation energies below -0.4 eV/atom- For mechanical performance: select A-site elements from the Ir/Pt/Rh group- For Co-base alternatives: Co₃(Ti,Al) mixtures show the best stability-lattice balance"""from pathlib import PathPath("../l12_design_hypotheses.md").write_text(hypotheses)print("Saved l12_design_hypotheses.md")